# Homework 4

In this dataset our desired target for classification task will be converted variable - has the client signed up to the platform or not.

Data preparation
Check if the missing values are presented in the features.
If there are missing values:
For categorical features, replace them with 'NA'
For numerical features, replace with with 0.0

Split the data into 3 parts with these exact calls:


df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

Question 1: ROC AUC feature importance
ROC AUC could also be used to evaluate feature importance of numerical variables.

Let's do that

For each numerical variable, use it as score (aka prediction) and compute the AUC with the y variable as ground truth.
Use the training dataset for that
If your AUC is < 0.5, invert this variable by putting "-" in front

(e.g. -df_train['balance'])

AUC can go below 0.5 if the variable is negatively correlated with the target variable. You can change the direction of the correlation by negating this variable - then negative correlation becomes positive.

Which numerical variable (among the following 4) has the highest AUC?

lead_score

number_of_courses_viewed

interaction_count

annual_income

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# 1. Load dataset
df = pd.read_csv('../data/course_lead_scoring_2026.csv')

# 2. Data preparation: handle missing values
for col in df.columns:
    if df[col].dtype == 'object' or df[col].dtype.name == 'category':
        df[col] = df[col].fillna('NA')
    else:
        df[col] = df[col].fillna(0.0)

# 3. Split the data into 3 parts with the exact specified calls
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

# 4. Compute ROC AUC feature importance for numerical variables
numerical_vars = ['lead_score', 'number_of_courses_viewed', 'interaction_count', 'annual_income']
y_train = df_train['converted'].values

for var in numerical_vars:
    score = df_train[var].values
    auc = roc_auc_score(y_train, score)
    
    # If AUC is < 0.5, invert the variable by putting a minus sign in front
    if auc < 0.5:
        auc = roc_auc_score(y_train, -score)
        
    print(f"ROC AUC for {var}: {auc:.3f}")

ROC AUC for lead_score: 0.788
ROC AUC for number_of_courses_viewed: 0.723
ROC AUC for interaction_count: 0.765
ROC AUC for annual_income: 0.608


Question 2: Training the model
Apply one-hot-encoding using DictVectorizer and train the logistic regression with these parameters:

LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
What's the AUC of this model on the validation dataset? (round to 3 digits)

0.532

0.632

0.732

0.832

In [3]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

target = 'converted'
y_val = df_val['converted'].values

# Exclude target and any ID column if present
features = [c for c in df.columns if c != target and c != 'id']

train_dicts = df_train[features].to_dict(orient='records')
val_dicts = df_val[features].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
val_auc = roc_auc_score(y_val, y_pred)

print(f"Validation AUC: {val_auc:.3f}")

Validation AUC: 0.732


Question 3: Precision and Recall
Now let's compute precision and recall for our model.

Evaluate the model on all thresholds from 0.0 to 1.0 with step 0.01
For each threshold, compute precision and recall
Plot them
At which threshold precision and recall curves intersect? Ignore thresholds at which both metrics are zero, and choose the threshold with the smallest absolute difference between precision and recall.

0.43


0.63

0.73

0.83

In [4]:
import numpy as np

thresholds = np.linspace(0.0, 1.0, 101)

scores = []
for t in thresholds:
    # Predict positive if probability >= threshold
    y_pred_bin = (y_pred >= t).astype(int)
    
    # True positives, false positives, false negatives
    tp = ((y_pred_bin == 1) & (y_val == 1)).sum()
    fp = ((y_pred_bin == 1) & (y_val == 0)).sum()
    fn = ((y_pred_bin == 0) & (y_val == 1)).sum()
    
    # Precision and Recall
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    scores.append((t, precision, recall))

# Filter out thresholds where both precision and recall are zero
filtered_scores = [s for s in scores if not (s[1] == 0 and s[2] == 0)]

# Find the threshold with the smallest absolute difference between precision and recall
best_threshold = min(filtered_scores, key=lambda x: abs(x[1] - x[2]))

print(f"Intersection/Closest threshold: {best_threshold[0]:.2f}")

Intersection/Closest threshold: 0.63


Question 4: F1 score
Precision and recall are conflicting - when one grows, the other goes down. That's why they are often combined into the F1 score - a metrics that takes into account both

This is the formula for computing F1:

Where 
P
 is precision and 
R
 is recall.

Let's compute F1 for all thresholds from 0.0 to 1.0 with increment 0.01

At which threshold F1 is maximal?

0.21

0.41

0.61

0.81

In [5]:
# Compute F1 score for each threshold and find the maximum
f1_scores = []
for t, precision, recall in scores:
    if (precision + recall) == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    f1_scores.append((t, f1))

best_f1_threshold = max(f1_scores, key=lambda x: x[1])
print(f"Threshold with maximal F1: {best_f1_threshold[0]:.2f}")

Threshold with maximal F1: 0.41


Question 5: 5-Fold CV
Use the KFold class from Scikit-Learn to evaluate our model on 5 different folds:

KFold(n_splits=5, shuffle=True, random_state=1)
Iterate over different folds of df_full_train
Split the data into train and validation
Train the model on train with these parameters: LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
Use AUC to evaluate the model on validation
How large is standard deviation of the scores across different folds?

0.001

0.007

0.013

0.060

In [6]:
from sklearn.model_selection import KFold

kfold = KFold(n_splits=5, shuffle=True, random_state=1)

scores = []
for train_idx, val_idx in kfold.split(df_full_train):
    df_train = df_full_train.iloc[train_idx]
    df_val = df_full_train.iloc[val_idx]
    
    y_train = df_train['converted'].values
    y_val = df_val['converted'].values
    
    features = [c for c in df_full_train.columns if c != 'converted' and c != 'id']
    
    train_dicts = df_train[features].to_dict(orient='records')
    val_dicts = df_val[features].to_dict(orient='records')
    
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(train_dicts)
    X_val = dv.transform(val_dicts)
    
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
    model.fit(X_train, y_train)
    
    y_pred = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, y_pred)
    scores.append(auc)

print(f"Std deviation: {np.std(scores):.3f}")

Std deviation: 0.007


Question 6: Hyperparameter Tuning
Now let's use 5-Fold cross-validation to find the best parameter C

Iterate over the following C values: [0.000001, 0.001, 1]
Initialize KFold with the same parameters as previously
Use these parameters for the model: LogisticRegression(solver='liblinear', C=C, max_iter=1000)
Compute the mean ROC AUC as well as the std (round the mean and std to 3 decimal digits).
Which C leads to the best mean ROC AUC?

0.000001

0.001

1

If you have ties, select the score with the lowest std. If you still have ties, select the smallest C.

In [7]:
n_splits = 5
kfold = KFold(n_splits=n_splits, shuffle=True, random_state=1)

C_values = [0.000001, 0.001, 1]

for C in C_values:
    scores = []
    for train_idx, val_idx in kfold.split(df_full_train):
        df_train = df_full_train.iloc[train_idx]
        df_val = df_full_train.iloc[val_idx]
        
        y_train = df_train['converted'].values
        y_val = df_val['converted'].values
        
        features = [c for c in df_full_train.columns if c != 'converted' and c != 'id']
        
        train_dicts = df_train[features].to_dict(orient='records')
        val_dicts = df_val[features].to_dict(orient='records')
        
        dv = DictVectorizer(sparse=False)
        X_train = dv.fit_transform(train_dicts)
        X_val = dv.transform(val_dicts)
        
        model = LogisticRegression(solver='liblinear', C=C, max_iter=1000)
        model.fit(X_train, y_train)
        
        y_pred = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, y_pred)
        scores.append(auc)
        
    mean_auc = np.mean(scores)
    std_auc = np.std(scores)
    print(f"C={C}: mean={mean_auc:.3f}, std={std_auc:.3f}")

C=1e-06: mean=0.617, std=0.019
C=0.001: mean=0.749, std=0.010
C=1: mean=0.732, std=0.007
